<a href="https://colab.research.google.com/github/obedglanson/senior_project_slc6a14/blob/main/3D_conf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install rdkit
!apt-get install openbabel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 41.4 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libboost-iostreams1.74.0 libinchi1 libmaeparser1 libopenbabel7
The following NEW packages will be installed:
  libboost-iostreams1.74.0 libinchi1 libmaeparser1 libopenbabel7 openbabel
0 upgraded, 5 newly installed, 0 to remove and 3 not upgraded.
Need to get 4,148 kB of archives.
After this operation, 19.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libboost-iostreams1.74.0 amd64 1.74.0-14ubuntu3 [245 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libinchi1 amd64 1.03+dfsg-4 [455 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libmaeparser1 amd64 1.2.4-1build1 [88.2 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libopenbabel7 amd64 3.1.1+dfsg-6ubuntu5 [3,23

In [2]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
import os
import subprocess

def apply_ph74_to_df(df, smiles_col, id_col, prefix):
    """
    runs OpenBabel in the background to protonate SMILES to pH 7.4,
    and maps them back to the DataFrame safely.
    """
    raw_smi = f"{prefix}_raw.smi"
    ph74_smi = f"{prefix}_ph74.smi"

    # 1. Write out strict .smi format for OpenBabel
    with open(raw_smi, "w") as f:
        for _, row in df.iterrows():
            s = str(row[smiles_col]).strip()
            i = str(row[id_col]).strip()
            f.write(f"{s}\t{i}\n")

    # 2. Execute OpenBabel via terminal subsystem
    print(f"Protonating {prefix}...")
    subprocess.run(["obabel", "-ismi", raw_smi, "-osmi", "-O", ph74_smi, "-p", "7.4"], capture_output=True)

    # 3. Read back in and map to IDs
    ph_dict = {}
    if os.path.exists(ph74_smi):
        with open(ph74_smi, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    ph_dict[parts[1]] = parts[0]

    # 4. Inject into dataframe, falling back to original SMILES if OpenBabel failed
    df['pH74_SMILES'] = df[id_col].astype(str).map(ph_dict)
    df['pH74_SMILES'] = df['pH74_SMILES'].fillna(df[smiles_col])

    return df


def process_and_embed_mols(dataframe, smiles_col, id_col, output_writer, num_confs=5):
    unique_molecules_saved = 0

    for index, row in dataframe.iterrows():
        smiles = row[smiles_col]

        # Extract ID
        if id_col in dataframe.columns and pd.notna(row[id_col]):
            mol_id = str(row[id_col])
        else:
            mol_id = f"Mol_{index}"

        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f"Warning: Cannot parse SMILES for {mol_id}. Skipping.")
            continue

        mol = Chem.AddHs(mol)

        params = AllChem.ETKDGv3()
        params.randomSeed = 42
        params.enforceChirality = True

        conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=num_confs, params=params)
        if not conf_ids:
            continue

        molecule_had_successful_conformer = False

        for conf_id in conf_ids:
            opt_status = AllChem.MMFFOptimizeMolecule(mol, confId=conf_id, maxIters=500)

            if opt_status == -1:
                print(f"Missing FF parameters for {mol_id}, Conf {conf_id}. Skipping.")
                continue
            elif opt_status == 1:
                pass # Did not fully converge, but geometry is still valid. Save it.

            # Unbundle
            single_conf_mol = Chem.Mol(mol)
            single_conf_mol.RemoveAllConformers()

            conf_copy = Chem.Conformer(mol.GetConformer(conf_id))
            conf_copy.SetId(0)

            single_conf_mol.AddConformer(conf_copy, assignId=False)

            # Strict Metadata
            single_conf_mol.SetProp("_Name", f"{mol_id}_conf_{conf_id}")
            single_conf_mol.SetProp("Parent_ID", mol_id)
            single_conf_mol.SetProp("Activity_Class", str(row.get('Activity_Class', 'Unknown')))

            output_writer.write(single_conf_mol)
            molecule_had_successful_conformer = True

        if molecule_had_successful_conformer:
            unique_molecules_saved += 1

    return unique_molecules_saved

# main pipeline

print("Loading datasets...")
sheet1_path = "/content/AA_Derivatives_Actives.csv"
sheet2_path = "/content/True_Negatives.csv"
sheet3_path = "/content/Enamine_VS_Hits_TrueNegs.csv"

df_sheet1_actives = pd.read_csv(sheet1_path)
df_neg = pd.read_csv(sheet2_path)
df_vs = pd.read_csv(sheet3_path)

print("Generating canonical SMILES representations to prevent data leakage...")
def safe_canonicalize(smiles_str):
    if pd.isna(smiles_str): return None
    mol = Chem.MolFromSmiles(str(smiles_str).strip())
    return Chem.MolToSmiles(mol, canonical=True) if mol is not None else None

df_sheet1_actives['Canonical_SMILES'] = df_sheet1_actives['SMILES'].apply(safe_canonicalize)
df_neg['Canonical_SMILES'] = df_neg['SMILES'].apply(safe_canonicalize)
df_vs['Canonical_SMILES'] = df_vs['SMILES'].apply(safe_canonicalize)

print("Enforcing activity definitions and handling missing data...")
extracted_ki = df_sheet1_actives['Exp Ki (µM)'].astype(str).str.split('±').str[0]
df_sheet1_actives['Numeric_Ki'] = pd.to_numeric(extracted_ki, errors='coerce')
df_sheet1_actives = df_sheet1_actives.dropna(subset=['Numeric_Ki', 'Canonical_SMILES']).copy()
df_sheet1_actives['Activity_Class'] = 1

valid_active_tiers = ['≤ 1 µM', '≤ 10 µM', '≤ 100 µM']
df_sheet3_actives = df_vs[df_vs['Activity Tier'].isin(valid_active_tiers)].copy()
df_sheet3_actives = df_sheet3_actives.dropna(subset=['Canonical_SMILES'])
df_sheet3_actives['Activity_Class'] = 1

df_vs_inactives = df_vs[df_vs['Activity Tier'] == 'No Inhibition'].copy()
df_vs_inactives = df_vs_inactives.dropna(subset=['Canonical_SMILES'])
df_vs_inactives['Activity_Class'] = 0

df_neg = df_neg.dropna(subset=['Canonical_SMILES']).copy()
df_neg['Activity_Class'] = 0

print("Executing Deduplication...")
sheet1_smiles = set(df_sheet1_actives['Canonical_SMILES'])
sheet3_active_smiles = set(df_sheet3_actives['Canonical_SMILES'])
all_active_smiles = sheet1_smiles | sheet3_active_smiles

df_sheet3_actives = df_sheet3_actives[~df_sheet3_actives['Canonical_SMILES'].isin(sheet1_smiles)]
df_neg = df_neg[~df_neg['Canonical_SMILES'].isin(all_active_smiles)]
df_vs_inactives = df_vs_inactives[~df_vs_inactives['Canonical_SMILES'].isin(all_active_smiles)]
df_vs_inactives = df_vs_inactives[~df_vs_inactives['Canonical_SMILES'].isin(set(df_neg['Canonical_SMILES']))]

# Protonate 2D SMILES before embedding
print("\nProtonating datasets to physiological pH 7.4...")
df_sheet1_actives = apply_ph74_to_df(df_sheet1_actives, 'Canonical_SMILES', 'Compound ID', 'train_actives_s1')
df_sheet3_actives = apply_ph74_to_df(df_sheet3_actives, 'Canonical_SMILES', 'Catalog ID', 'train_actives_s3')
df_neg = apply_ph74_to_df(df_neg, 'Canonical_SMILES', 'Catalog ID', 'train_neg_s2')
df_vs_inactives = apply_ph74_to_df(df_vs_inactives, 'Canonical_SMILES', 'Catalog ID', 'train_neg_s3')


print("\nData deduplication & protonation complete. Proceeding to 3D embedding...")

# 3D Embedding and Export
writer_train_actives = Chem.SDWriter("Training_Actives_3D.sdf")
act_count_1 = process_and_embed_mols(df_sheet1_actives, 'pH74_SMILES', 'Compound ID', writer_train_actives, num_confs=5)
act_count_2 = process_and_embed_mols(df_sheet3_actives, 'pH74_SMILES', 'Catalog ID', writer_train_actives, num_confs=5)
writer_train_actives.close()

total_actives = act_count_1 + act_count_2
print(f"Set 1 Complete: {total_actives} unique molecules saved for Training (ACTIVES).")

writer_train_negatives = Chem.SDWriter("Training_Negatives_3D.sdf")
neg_count_s2 = process_and_embed_mols(df_neg, 'pH74_SMILES', 'Catalog ID', writer_train_negatives, num_confs=5)
neg_count_s3 = process_and_embed_mols(df_vs_inactives, 'pH74_SMILES', 'Catalog ID', writer_train_negatives, num_confs=5)
writer_train_negatives.close()

total_negatives = neg_count_s2 + neg_count_s3
print(f"Set 2 Complete: {total_negatives} unique molecules saved for Training (NEGATIVES).")

Loading datasets...
Generating canonical SMILES representations to prevent data leakage...
Enforcing activity definitions and handling missing data...
Executing Deduplication...

Protonating datasets to physiological pH 7.4...
Protonating train_actives_s1...
Protonating train_actives_s3...
Protonating train_neg_s2...
Protonating train_neg_s3...

Data deduplication & protonation complete. Proceeding to 3D embedding...
Set 1 Complete: 83 unique molecules saved for Training (ACTIVES).
Set 2 Complete: 54 unique molecules saved for Training (NEGATIVES).
